# 03 — Complexity cues, controller with hysteresis, random and oracle baselines (calib)

**Purpose.** Compute the per-frame cues of PLAN §3.1 on the 20 calib clips:
- image cues: motion, clutter (Canny), HSV histogram distance;
- detection cues: count and uncertainty, taken from the detections of the level actually chosen
  at t − 1.

Then replay the INT8/FP16 controller (EMA, threshold, K-frame persistence, scene-change override)
on **notebook 02's cached detections** and compare it with fixed FP16/INT8, matched-random
(Markov, 10 seeds), i.i.d.-random (10 seeds) and the oracle.

This is **development on calib only**: the controller uses default parameters, with normalisation
and θ fit on calib. Calibration and locking come in notebook 04b. **The test split is not touched.**

Built-in integrity checks:
- The attached cache must reproduce the committed per-frame detection counts exactly.
- Fixed FP16/INT8 and the oracle, re-scored by the composed evaluator, must equal notebook 02's
  committed mAP on every slice (to 1e-9).

**Before running**
1. *Add Data* → **`tcjpaperdatasets`** (images, for the cues).
2. *Add Data* → *Your Datasets* → **`adapres-cache-02`**. This is your private dataset made from
   `dist/adapres_cache_02.zip`: notebook 02's detection cache with a SHA-256 manifest; see the repo
   README, "Detection cache dataset". Attaching notebook 02's output directly also works, but attach
   only one source. Cell 3 rejects any cache that doesn't reproduce the committed detection counts.
3. *Settings* → **Accelerator: GPU T4 x2** (convention; this notebook is CPU-bound), **Internet: On**.
   *Add-ons → Secrets* → tick **`GH_TOKEN`** for this notebook.
4. `SMOKE_TEST = True` first (2 clips, ≈6 min including installs). Then `SMOKE_TEST = False` →
   *Save Version → Save & Run All* (20 clips, ≈15–20 min; the estimate is unverified).

**Download afterwards:** `results_03_controller_dev.zip` → unzip into the repo root
(`results/bdd100k_mot/calib_*_dev/`, `calib_features_yolo11s/`, `results/sanity/sanity_03_yolo11s/`,
`logs/`). Don't download the working folder.


In [ ]:
# ===================== CELL 1: CONFIG + ENVIRONMENT CHECK =====================
# Edit nothing below for a normal run except SMOKE_TEST.
SMOKE_TEST = True   # True: tiny subset (run this first). False: full run.
NOTEBOOK_REV = "f7a8f746ac3c"   # hash of this notebook's generated source; checked against the cloned repo

CONFIG = {
    "notebook": "03_controller_dev",
    "repo_https": "https://github.com/vinaykamble-nsio/adapres.git",       # private repo, cloned with Secret GH_TOKEN
    "repo_ref": "main",
    "repo_dir": "/tmp/adapres",                     # /tmp: not saved into notebook output
    "repo_zip_marker": "adapres/GIT_COMMIT",        # fallback: repo zip attached as a dataset
    "config_path": "configs/default.yaml",          # relative to repo_dir
    "overrides": {                                  # applied on top of default.yaml
        "paths": {"input_root": "/kaggle/input", "work_root": "/kaggle/working"},
    },
    "cuda_visible_devices": "0",                    # T4 x2 -> pin to one GPU
    "required_gpu_substring": "T4",
}

import os, sys, subprocess, json, platform, base64, glob, shutil, time
os.environ["CUDA_VISIBLE_DEVICES"] = CONFIG["cuda_visible_devices"]  # before any CUDA import
T0 = time.time()

def sh(cmd, check=True):
    """Run a shell command, print its output, raise with the real error on failure."""
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.stderr.strip():
        print(r.stderr[-6000:], file=sys.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

sh("nvidia-smi")
print("python", sys.version)
for mod in ("torch", "onnxruntime", "tensorrt", "ultralytics"):
    try:
        m = __import__(mod)
        print(f"{mod} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{mod} NOT AVAILABLE YET ({e!r}); installed in cell 2 if pinned")
import torch
print("torch.cuda.is_available:", torch.cuda.is_available(), "| visible devices:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
gpu = torch.cuda.get_device_name(0)
print("GPU 0:", gpu)
if CONFIG["required_gpu_substring"] not in gpu:
    raise RuntimeError(f"GPU is {gpu!r}; this project requires a T4 (P100 has no fast INT8). "
                       "Settings -> Accelerator -> GPU T4 x2, then restart.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"expected exactly 1 visible GPU after pinning, got {torch.cuda.device_count()}")

# ---- get the repo: private GitHub clone (Secret GH_TOKEN), else attached zip dataset
REPO = CONFIG["repo_dir"]
repo_source = None
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
try:
    if "github.com/OWNER/" in CONFIG["repo_https"]:
        raise RuntimeError("repo_https is still the placeholder: set it in configs/kaggle.yaml and regenerate")
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    if not token:
        raise RuntimeError("Secret GH_TOKEN is empty")
    hdr = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    # token passed as a one-off header: never written to .git/config or printed
    r = subprocess.run(["git", "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {hdr}",
                        "clone", "--quiet", CONFIG["repo_https"], REPO], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed: " + r.stderr.replace(token, "***").replace(hdr, "***"))
    sh(["git", "-C", REPO, "checkout", "--quiet", CONFIG["repo_ref"]])
    repo_source = "github"
    del token, hdr
except Exception as e:
    print(f"GitHub clone unavailable: {e!r}\nTrying zip-dataset fallback ...")
    hits = sorted(glob.glob("/kaggle/input/**/" + CONFIG["repo_zip_marker"], recursive=True))
    if not hits:
        raise RuntimeError("Neither GitHub clone (Secret GH_TOKEN) nor an attached repo zip "
                           "(dataset containing adapres/GIT_COMMIT) is available. See README cell.") from e
    shutil.copytree(os.path.dirname(hits[0]), REPO)
    repo_source = "zip_dataset:" + os.path.dirname(hits[0])

sys.path.insert(0, os.path.join(REPO, "src"))
sys.path.insert(0, os.path.join(REPO, "scripts"))
# Re-running cells in a live kernel re-clones the repo but Python would keep the OLD imported
# modules while GIT records the NEW commit. Drop them so the code always matches the hash.
_stale = [m for m in list(sys.modules) if m == "adapres" or m.startswith("adapres.") or m == "make_notebooks"]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"dropped {len(_stale)} previously imported adapres modules (fresh code from this clone)")
import make_notebooks
expected_rev = make_notebooks.notebook_rev(CONFIG["notebook"] + ".ipynb")
if expected_rev != NOTEBOOK_REV:
    raise RuntimeError(f"STALE NOTEBOOK: this notebook is rev {NOTEBOOK_REV} but repo {CONFIG['repo_ref']} "
                       f"generates rev {expected_rev}. Re-import notebooks/{CONFIG['notebook']}.ipynb "
                       "from the repo (File -> Import Notebook), or push your local commits.")
print("notebook rev", NOTEBOOK_REV, "matches repo")
from adapres import config as C
from adapres.repro import git_commit, set_seed
cfg = C.load(os.path.join(REPO, CONFIG["config_path"]), CONFIG["overrides"])
SEED = cfg["seed"]
set_seed(SEED)
GIT = git_commit(REPO)
print("repo source:", repo_source)
print("git commit:", GIT)
print("seed:", SEED, "| SMOKE_TEST:", SMOKE_TEST)
if GIT["commit"] is None:
    raise RuntimeError("could not determine the git commit; refusing to produce unattributable results")
if GIT["dirty"]:
    raise RuntimeError("repo checkout is dirty; results would not match the recorded commit")
RESULTS = C.resolve_out(cfg, "results_dir", SMOKE_TEST)
LOGS = C.resolve_out(cfg, "logs_dir", SMOKE_TEST)
# Start every run with empty results/ and logs/: the final zip packs these folders, and leftovers
# from an earlier run in the same session would otherwise be shipped as if produced now.
# (Resumable caches live in work dirs, not here.)
for d in (RESULTS, LOGS):
    if os.path.isdir(d) and os.listdir(d):
        print(f"clearing stale outputs from an earlier run in this session: {d} ({len(os.listdir(d))} entries)")
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
print("results ->", RESULTS, "| logs ->", LOGS)


In [ ]:
# ===================== CELL 2: LOCKED INSTALLS + VERSION CHECK =====================
# Internet must be ON. Installs only what the Kaggle image lacks; torch is not touched.
# Every pin is exact (locked 2026-11-20); the cell fails if anything installs at another version.
import re, importlib.metadata as md
print("python", sys.version, "| torch", torch.__version__, "| torch CUDA", torch.version.cuda)
REQ = os.path.join(REPO, "requirements-kaggle.txt")
lines = [ln.split("#")[0].strip() for ln in open(REQ)]
opts = [tok for ln in lines if ln.startswith("-") for tok in ln.split()]  # e.g. --extra-index-url URL
print("pip options:", opts)
reqs = [ln for ln in lines if ln and not ln.startswith("-")]
PIP = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check"]
# Preflight: resolve each requirement on its own (no install) so ALL failures show in one run.
bad = {}
for req in reqs:
    p = subprocess.run(PIP + ["--dry-run", "--no-deps", "--quiet", *opts, req], capture_output=True, text=True)
    if p.returncode != 0:
        bad[req] = "\n".join(l for l in p.stderr.splitlines() if "ERROR" in l) or p.stderr[-2000:]
    print(f"preflight {req:45s} {'OK' if p.returncode == 0 else 'FAILED'}")
if bad:
    for req, err in bad.items():
        print(f"\n--- {req} ---\n{err}")
    raise RuntimeError(f"{len(bad)} requirement(s) have no wheel for this Python/CUDA: {list(bad)}. "
                       "Paste this whole cell output; the available versions are listed above.")
r = sh(PIP + ["--quiet", "-r", REQ], check=False)
with open(os.path.join(LOGS, f"{CONFIG['notebook']}_pip.log"), "w") as f:
    f.write(r.stdout + "\n--- stderr ---\n" + r.stderr)
if r.returncode != 0:
    raise RuntimeError("pip install failed (real error above).")
importlib.invalidate_caches()
# Version lock: record exactly what got installed so requirements-kaggle.txt can be pinned to it.
names = [re.split(r"[<>=!~\[ ]", q)[0] for q in reqs]
resolved = {n: md.version(n) for n in names}
os.makedirs(os.path.join(RESULTS, "env"), exist_ok=True)
lock = "\n".join(f"{n}=={v}" for n, v in resolved.items())
with open(os.path.join(RESULTS, "env", "pip_resolved.txt"), "w") as f:
    f.write(f"# python {sys.version.split()[0]} torch {torch.__version__} cuda {torch.version.cuda}\n{lock}\n")
print("RESOLVED VERSIONS (copy into requirements-kaggle.txt as exact pins):\n" + lock)
trt_dists = ("tensorrt-cu13", "tensorrt-cu13-bindings", "tensorrt-cu13-libs")
if len({resolved.get(d) for d in trt_dists}) != 1:
    raise RuntimeError(f"TensorRT wrapper/bindings/libs version mismatch: {resolved}")
pinned = {re.split(r"[<>=!~\[ ]", q)[0]: q.split("==")[-1] for q in reqs if "==" in q}
drift = {n: (pinned[n], resolved[n]) for n in pinned if pinned[n] != resolved[n]}
if drift:
    raise RuntimeError(f"installed versions differ from the lock (pinned, installed): {drift}")
try:
    import tensorrt
except ImportError:
    # Diagnose instead of guessing: which top-level modules did each TensorRT dist install?
    for d in trt_dists:
        try:
            tops = sorted({str(f).split("/")[0] for f in (md.files(d) or [])})
            print(f"{d}: top-level entries {tops[:25]}")
        except md.PackageNotFoundError:
            print(f"{d}: NOT INSTALLED")
    raise
import onnx, onnxruntime
print("import OK: tensorrt", tensorrt.__version__, "| onnx", onnx.__version__, "| onnxruntime", onnxruntime.__version__)
print("torch still", torch.__version__, "cuda", torch.version.cuda)


In [ ]:
# ===================== CELL 3: PREPARE + LOCATE AND VERIFY THE NOTEBOOK 02 CACHE =====================
import logging
logging.basicConfig(level=logging.INFO, force=True, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(os.path.join(LOGS, "03_controller.log"))])
import cv2
print("opencv", cv2.__version__)
from adapres.pipeline.controller_dev import Step3
t = time.time()
st = Step3(cfg, SMOKE_TEST, REPO)
info = st.prepare()
print(json.dumps({k: info[k] for k in ("n_frames", "slice_sizes")}, indent=1), "| sequences:", len(info["sequences"]))
root = st.locate_cache()
print("cache:", root)
vc = st.verify_cache()
print("cache verification:", vc)
print(f"prepare: {time.time()-t:.0f}s")


In [ ]:
# ===================== CELL 4: IMAGE CUES (checkpointed per clip) =====================
t = time.time()
ft = st.features()
print(json.dumps(ft, indent=1))
print(f"features: {(time.time()-t)/60:.1f} min")


In [ ]:
# ===================== CELL 5: NORMALISATION + DEV THRESHOLD (calib) =====================
fit = st.fit()
print("norm (p5, p95):", fit["norm"])
print("theta (quantile %.2f of open-loop S_hat): %.4f" % (fit["theta_quantile"], fit["theta"]))
print("frames with scene-change cue (d >= theta_sc):", round(fit["frac_frames_scene_cue"], 4))
print(json.dumps(fit["cue_percentiles"], indent=1))


In [ ]:
# ===================== CELL 6: POLICIES (replay + composed pycocotools eval) =====================
import pandas as pd
t = time.time()
R = st.policies()
rows = []
for name, m in R.items():
    a = m["all"]
    r = {"policy": name, "mAP50-95": 100*a["map50_95"], "sd": 100*a.get("map50_95_sd", 0) if "map50_95_sd" in a else None,
         "mAP50": 100*a["map50"], "INT8 frac": a["frac_level"].get("int8"), "switch/100": a.get("switch_rate_per_100"),
         "est engine ms": a.get("est_engine_ms_mean")}
    for s in ("easy", "crowded", "night"):
        r[f"{s}"] = 100*m[s]["map50_95"] if m[s]["n_images"] else None
        r[f"{s} INT8"] = m[s]["frac_level"].get("int8") if m[s]["n_images"] else None
    rows.append(r)
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("controller step time:", R[f"adaptive_K{cfg['controller']['K']}"]["all"]["ctrl_step_ms"])
print("NOTE: calib development numbers with default (unlocked) controller parameters; est engine ms = microbench medians.")
print(f"policies: {(time.time()-t)/60:.1f} min")


In [ ]:
# ===================== CELL 7: SANITY CHECKS =====================
SAN = st.sanity()
for c in SAN["checks"]:
    print(f"[{'PASS' if c['pass'] else 'FAIL'}] {c['check']}  {c['detail']}")
print("informational:", SAN["informational"])
print("ALL SANITY CHECKS PASS" if SAN["all_pass"] else "\n*** SANITY FAILURES ABOVE (results saved) ***")


In [ ]:
# ===================== FINAL CELL: ZIP RESULTS =====================
import zipfile
suffix = "_smoke" if SMOKE_TEST else ""
zpath = f"/kaggle/working/results_03_controller_dev{suffix}.zip"
base = os.path.dirname(RESULTS)  # contains results/ and logs/
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in (RESULTS, LOGS):
        for root, _, files in os.walk(top):
            for fn in files:
                full = os.path.join(root, fn)
                z.write(full, os.path.relpath(full, base))
                n += 1
print(f"wrote {zpath} ({n} files, {os.path.getsize(zpath)/1e6:.2f} MB)")
print(f"total notebook time: {(time.time()-T0)/60:.1f} min")
if not SAN["all_pass"]:
    raise RuntimeError("sanity check(s) failed (zip written): " + str([c["check"] for c in SAN["checks"] if not c["pass"] and c.get("binding", True)]))
